# Notebook 2: Custom NER Training with spaCy

Pre-trained models are great, but what if you need to recognize **custom entity types**?
(e.g., product codes, medical terms, internal jargon)

## What you'll learn:
1. Preparing training data in spaCy format
2. Converting data to spaCy's DocBin format
3. Training a custom NER model from a blank pipeline
4. Updating an existing model with new entity types
5. Evaluating your trained model

## 1. Training Data Format

spaCy expects training data as a list of tuples:
```python
TRAIN_DATA = [
    ("text string", {"entities": [(start, end, "LABEL"), ...]}),
    ...
]
```

We'll build a model that recognizes **TECH_SKILL** and **JOB_TITLE** in job postings.

In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.training import Example
import random
import json

print(f"spaCy version: {spacy.__version__}")

In [ ]:
# Training data: job postings with TECH_SKILL and JOB_TITLE annotations
TRAIN_DATA = [
    ("We need a Senior Data Scientist with experience in Python and TensorFlow.",
     {"entities": [(10, 31, "JOB_TITLE"), (52, 58, "TECH_SKILL"), (63, 73, "TECH_SKILL")]}),
    
    ("Looking for a Machine Learning Engineer who knows PyTorch and Kubernetes.",
     {"entities": [(14, 40, "JOB_TITLE"), (51, 58, "TECH_SKILL"), (63, 73, "TECH_SKILL")]}),
    
    ("The Full Stack Developer role requires React, Node.js, and PostgreSQL.",
     {"entities": [(4, 24, "JOB_TITLE"), (40, 45, "TECH_SKILL"), (47, 54, "TECH_SKILL"), (60, 70, "TECH_SKILL")]}),
    
    ("Hiring a DevOps Engineer with expertise in Docker and AWS.",
     {"entities": [(9, 24, "JOB_TITLE"), (44, 50, "TECH_SKILL"), (55, 58, "TECH_SKILL")]}),
    
    ("Our Backend Developer should be proficient in Java and Spring Boot.",
     {"entities": [(4, 21, "JOB_TITLE"), (46, 50, "TECH_SKILL"), (55, 66, "TECH_SKILL")]}),
    
    ("Seeking a Frontend Developer skilled in TypeScript and Vue.js.",
     {"entities": [(10, 28, "JOB_TITLE"), (40, 50, "TECH_SKILL"), (55, 61, "TECH_SKILL")]}),
    
    ("The Data Analyst position requires proficiency in SQL and Tableau.",
     {"entities": [(4, 16, "JOB_TITLE"), (52, 55, "TECH_SKILL"), (60, 67, "TECH_SKILL")]}),
    
    ("We are looking for a Cloud Architect experienced with Azure and Terraform.",
     {"entities": [(21, 36, "JOB_TITLE"), (54, 59, "TECH_SKILL"), (64, 73, "TECH_SKILL")]}),
    
    ("The Software Engineer must know C++ and Linux.",
     {"entities": [(4, 21, "JOB_TITLE"), (32, 35, "TECH_SKILL"), (40, 45, "TECH_SKILL")]}),
    
    ("Hiring a Product Manager with knowledge of Agile and Jira.",
     {"entities": [(9, 24, "JOB_TITLE"), (44, 49, "TECH_SKILL"), (54, 58, "TECH_SKILL")]}),
]

print(f"Training examples: {len(TRAIN_DATA)}")
print(f"\nSample:")
print(f"  Text: {TRAIN_DATA[0][0]}")
print(f"  Entities: {TRAIN_DATA[0][1]['entities']}")

## 2. Validate Training Data

Always validate that your character offsets are correct before training!

In [ ]:
# Validate that annotations match the text
print("Validating training data...")
print("=" * 60)
for i, (text, annotations) in enumerate(TRAIN_DATA):
    for start, end, label in annotations["entities"]:
        span = text[start:end]
        print(f"  [{label:12s}] '{span}'")
        # Check for whitespace issues
        if span != span.strip():
            print(f"    WARNING: Leading/trailing whitespace in example {i}!")
print("\nValidation complete!")

## 3. Training a Blank NER Model

We'll train a model from scratch with **only** our custom entity types.

### Steps:
1. Create a blank English model
2. Add the NER component
3. Add custom labels
4. Train with our data

In [ ]:
# Create a blank model and add NER
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

# Add custom labels
ner.add_label("TECH_SKILL")
ner.add_label("JOB_TITLE")

print(f"Pipeline: {nlp.pipe_names}")
print(f"NER labels: {ner.labels}")

In [ ]:
# Train the model
optimizer = nlp.begin_training()

# Training loop
n_epochs = 30
losses_history = []

for epoch in range(n_epochs):
    random.shuffle(TRAIN_DATA)
    losses = {}
    
    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], drop=0.3, sgd=optimizer, losses=losses)
    
    losses_history.append(losses.get("ner", 0))
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs}  |  Loss: {losses.get('ner', 0):.4f}")

print("\nTraining complete!")

In [ ]:
import matplotlib.pyplot as plt

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(range(1, n_epochs + 1), losses_history, color="steelblue", linewidth=2)
plt.title("NER Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Test the Trained Model

In [ ]:
from spacy import displacy

# Test on new, unseen text
test_texts = [
    "We need a Data Engineer with experience in Spark and Kafka.",
    "The Machine Learning Engineer should know Python and scikit-learn.",
    "Looking for a Frontend Developer who is skilled in React and CSS.",
]

for text in test_texts:
    doc = nlp(text)
    print(f"\nText: {text}")
    print("Entities:")
    for ent in doc.ents:
        print(f"  [{ent.label_:12s}] '{ent.text}'")
    if not doc.ents:
        print("  (no entities found)")
    print()

In [ ]:
# Visualize with displaCy
colors = {"TECH_SKILL": "#2ecc71", "JOB_TITLE": "#3498db"}
options = {"colors": colors}

doc = nlp("We need a Senior Data Scientist with experience in Python and TensorFlow.")
displacy.render(doc, style="ent", jupyter=True, options=options)

## 5. Saving and Loading the Model

In [ ]:
# Save the model to disk
output_dir = "../models/custom_ner_jobs"
nlp.to_disk(output_dir)
print(f"Model saved to: {output_dir}")

# Load it back
nlp_loaded = spacy.load(output_dir)
doc = nlp_loaded("Hiring a Backend Developer with Java and Spring Boot experience.")
print("\nLoaded model test:")
for ent in doc.ents:
    print(f"  [{ent.label_:12s}] '{ent.text}'")

## 6. Adding New Entities to an Existing Model

You can also add your custom labels to a **pre-trained** model (like `en_core_web_sm`) so it keeps its existing knowledge while learning your new entities.

**Important**: You must provide examples of BOTH old and new entity types to prevent catastrophic forgetting.

In [ ]:
# Load pre-trained model
nlp2 = spacy.load("en_core_web_sm")
ner2 = nlp2.get_pipe("ner")

# Add our new label
ner2.add_label("TECH_SKILL")
ner2.add_label("JOB_TITLE")

print(f"Labels (now includes custom): {ner2.labels}")
print(f"\nNote: To properly fine-tune, you would need a larger dataset")
print(f"that includes examples of the original entity types too.")
print(f"Otherwise the model will 'forget' how to detect PERSON, ORG, etc.")

## 7. Converting Data to DocBin (spaCy v3 Best Practice)

For production training with `spacy train` CLI, you should convert your data to `.spacy` (DocBin) format.

In [ ]:
# Convert training data to DocBin format
nlp_blank = spacy.blank("en")
doc_bin = DocBin()

for text, annotations in TRAIN_DATA:
    doc = nlp_blank.make_doc(text)
    ents = []
    for start, end, label in annotations["entities"]:
        span = doc.char_span(start, end, label=label)
        if span is not None:
            ents.append(span)
        else:
            print(f"WARNING: Skipping entity ({start}, {end}, {label}) - misaligned span")
    doc.ents = ents
    doc_bin.add(doc)

# Save as .spacy file
doc_bin.to_disk("../data/train.spacy")
print(f"Saved {len(TRAIN_DATA)} docs to ../data/train.spacy")
print(f"\nYou can now use this with: python -m spacy train config.cfg --paths.train ../data/train.spacy")

## 8. Evaluation

Let's evaluate our model properly using held-out test data.

In [ ]:
# Hold-out test data
TEST_DATA = [
    ("We need a Senior Data Scientist with experience in Python and TensorFlow.",
     {"entities": [(10, 31, "JOB_TITLE"), (52, 58, "TECH_SKILL"), (63, 73, "TECH_SKILL")]}),
    ("Hiring a DevOps Engineer with expertise in Docker and AWS.",
     {"entities": [(9, 24, "JOB_TITLE"), (44, 50, "TECH_SKILL"), (55, 58, "TECH_SKILL")]}),
]

# Evaluate
examples = []
for text, annotations in TEST_DATA:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annotations)
    examples.append(example)

scores = nlp.evaluate(examples)
print("Evaluation Results:")
print(f"  Entity Precision: {scores['ents_p']:.2f}")
print(f"  Entity Recall:    {scores['ents_r']:.2f}")
print(f"  Entity F1:        {scores['ents_f']:.2f}")

# Per-entity-type scores
if 'ents_per_type' in scores:
    print("\nPer-entity scores:")
    for ent_type, ent_scores in scores['ents_per_type'].items():
        print(f"  {ent_type:12s}  P={ent_scores['p']:.2f}  R={ent_scores['r']:.2f}  F1={ent_scores['f']:.2f}")

## 9. Tips for Custom NER

1. **More data = better**: Aim for 200+ annotated examples minimum
2. **Include negative examples**: Texts with NO entities help the model learn what's NOT an entity
3. **Consistent labeling**: Same entity = same label everywhere
4. **Context matters**: Include varied sentence structures
5. **Use annotation tools**: [Prodigy](https://prodi.gy/), [Label Studio](https://labelstud.io/), or [Doccano](https://doccano.github.io/)

---

**Next**: [Notebook 3 - NER with Transformers (BERT)](03_ner_with_transformers.ipynb)